# Commande utiles shell

In [ ]:
# compter des lignes

wc -l test.txt

# chercher un mot

grep "rs123" fichier.txt

# Décompresser a la volée 

zcat chr11.vcf.gz | head



# Type fichier

## VCF

un ficher **VCF** contient des variants génétiques, souvent : SNP, indels, génotypes pour plusieurs individus

exemple : 

```

##fileformat=VCFv4.2
##source=bcftools
#CHROM  POS     ID       REF ALT QUAL FILTER INFO FORMAT sample1 sample2
1       10583   rs1      G   A   29   PASS   .    GT     0/1     0/0
1       10611   rs2      C   G   45   PASS   .    GT     1/1     0/1

```

Les lignes commencent par # sont le header, 2 types :

- Métadonnées ##
- Lignes principales des colonnes #

### Les colonnes a connaitre : 

CHROM -> Chromosome

POS -> Position sur le chromosome

ID -> Nom du variant, souvent un rsID

REF -> Allèle de référence

ALT -> Allèle alternatif

QUAL -> Qualité du variant

FILTER -> Statut du filtre
    exemple : PASS = OK  |  autre chose = variant suspect ou filtré

INFO -> Infos complémentaires

FORMAT -> Indique ce que contiennent les colonnes échantillons


### Comprendre les génotypes :

0/0 -> homozygote référence

0/1 -> hétérozygote

1/1 -> homozygote alternatif

./. -> génotype manquant

### Ce qu’on filtre souvent dans un VCF

En pratique, on retire souvent :

- les variants trop rares selon l’objectif
- les variants avec trop de données manquantes
- les variants de mauvaise qualité
- les variants multialléliques selon le pipeline
- les indels si on veut un GWAS SNP simple

# Concept Génétique

## MAF

Fréquence de l'allèle minoritaire 

Par exemple : A = 95% G = 5%
Alors : MAF = 0.05

## Missingness

C'est le taux de données manquantes

Il y a 2 côtés : 
- le missingness par variant -> SNP mal génotypé dans bcp d'individus 
- le missingess par individu -> un individu avec bcp de SNPs manquants 

En pratique on filtre souvent : 
- variants : geno 0.05
- individus : mind 0.05

## HWE 

**Hardy-Weinberg Equilibrium**

La loi de Hardy-Weinberg stipule que les fréquences des allèles et des génotypes restent constantes dans une population idéale. 

Cette loi s’applique seulement si une population est grande, qu’il n’y a ni mutation, ni migration, ni sélection naturelle ou sexuelle. 

Les deux équations fondamentales sont p + q = 1 (pour les fréquences alléliques) et p² + 2pq + q² = 1 (pour les fréquences génotypiques).
En connaissant la fréquence d’un phénotype récessif (q²), il est possible de calculer les fréquences des autres allèles et génotypes. 
Le principe sert de référence pour les scientifiques afin de déterminer si une population est en cours d’évolution.

En savoir plus sur: https://jeretiens.net/le-principe-dequilibre-de-hardy-weinberg/


test statistique pour détecter des variants bizarres  avec : HWE p < 1e-6



## LD 

**Linkage Disequilibrium**  -> Situation dans laquelle deux gènes sont trouvés ensemble dans une population à une fréquence supérieure à celle prédite par le produit de leur fréquence individuelle

2 variant proche peuvent être corrélés et dans certains cas on veut évité d'avoir trop de SNPs redondants, donc on fait du LD pruning 


# bcftools 

## Lecture VCF

In [ ]:
# voir header (-h --header-only)

bcftools view -h files.vcf.gz 
bcftools view -h files.vcf.g | less # car souvent trop long



# voir uniquement les variants (-H --no-header)

bcftools view -H files.vcf.gz 
bcftools view -H files.vcf.gz | head -5 # si que les 5 premiers 

# lister les samples (-l list-samples)

bcftools query -l files.vcf.gz 

# extraire colonnes 

bcftools query -f 'FORMAT' file.vcf.gz 
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' files.vcf.gz

# compter les samples 

bcftools query -l files.vcf.gz | wc -l 

# compter les variants

bcftools view -H files.vcf.gz | wc -l 

# voir les chromosomes présents 

bcftools query -f '%CHROM\n' files.vcf.gz | sort | uniq 